# 05 — Ensembles Base

Este notebook treina e otimiza os **classificadores de ensemble** do TCC nas 4 representações textuais.

## Ideia central dos Ensembles

Em vez de depender de um único modelo, ensembles **combinam múltiplos classificadores** para obter uma previsão mais robusta e precisa. Há duas estratégias principais:

| Estratégia | Como funciona | Reduz | Modelos |
|---|---|---|---|
| **Bagging** | Treina N modelos em paralelo em subamostras aleatórias do treino | Variância | Random Forest |
| **Boosting** | Treina N modelos em sequência, cada um corrigindo os erros do anterior | Bias | AdaBoost, Gradient Boosting, XGBoost |

## Classificadores abordados:
1. **Random Forest** — Bagging de múltiplas Árvores de Decisão
2. **AdaBoost** — Boosting adaptativo por reponderação de amostras
3. **Gradient Boosting** — Boosting por descida de gradiente
4. **XGBoost** — Boosting por gradiente otimizado com regularização

> **Pré-requisito:** Execute o Notebook 03 para gerar os splits em `data/processed/`.

## 1. Importações e Carregamento dos Dados

In [1]:
import sys
sys.path.append('..')  # Permite importar módulos da pasta src/ a partir de notebooks/

import os
import joblib
import numpy as np
import pandas as pd

from src.models import (
    criar_random_forest,         # Fábrica: RandomForestClassifier configurado com n_jobs=-1
    criar_adaboost,              # Fábrica: AdaBoostClassifier com algoritmo SAMME
    criar_gradient_boosting,     # Fábrica: GradientBoostingClassifier sklearn
    criar_xgboost,               # Fábrica: XGBClassifier com suporte a matrizes esparsas
    PARAMS_RANDOM_FOREST,        # Grid: n_estimators, max_depth, min_samples_split, max_features
    PARAMS_ADABOOST,             # Grid: n_estimators, learning_rate
    PARAMS_GRADIENT_BOOSTING,    # Grid: n_estimators, learning_rate, max_depth
    PARAMS_XGBOOST,              # Grid: n_estimators, max_depth, learning_rate, subsample
    otimizar_modelo,             # GridSearchCV com StratifiedKFold(5) e f1_weighted
    salvar_modelo,               # Persiste o best_estimator_ em disco (.joblib)
    para_denso,                  # Converte matriz esparsa para numpy array denso
)

os.makedirs('../results/metrics', exist_ok=True)

print('✅ Importações concluídas com sucesso!')

✅ Importações concluídas com sucesso!


In [2]:
# Carrega os 4 splits gerados no Notebook 03.
# As labels y são as mesmas em todos os splits — apenas as features X mudam.
X_train_bow,   X_test_bow,   y_train, y_test = joblib.load('../data/processed/splits_bow.joblib')
X_train_tfidf, X_test_tfidf, _,       _      = joblib.load('../data/processed/splits_tfidf.joblib')
X_train_w2v,   X_test_w2v,   _,       _      = joblib.load('../data/processed/splits_w2v.joblib')
X_train_glove, X_test_glove, _,       _      = joblib.load('../data/processed/splits_glove.joblib')

print(f'BoW    — Treino: {X_train_bow.shape}   | Teste: {X_test_bow.shape}')
print(f'TF-IDF — Treino: {X_train_tfidf.shape} | Teste: {X_test_tfidf.shape}')
print(f'W2V    — Treino: {X_train_w2v.shape}   | Teste: {X_test_w2v.shape}')
print(f'GloVe  — Treino: {X_train_glove.shape} | Teste: {X_test_glove.shape}')
print(f'\nDistribuição das classes no treino (0=sem ideação | 1=com ideação):')
print(y_train.value_counts().to_string())

BoW    — Treino: (3021, 3875)   | Teste: (756, 3875)
TF-IDF — Treino: (3021, 5000) | Teste: (756, 5000)
W2V    — Treino: (3021, 100)   | Teste: (756, 100)
GloVe  — Treino: (3021, 100) | Teste: (756, 100)

Distribuição das classes no treino (0=sem ideação | 1=com ideação):
label
0    2149
1     872


## 2. Random Forest

O **Random Forest** é um ensemble de Bagging que treina N Árvores de Decisão **independentes** em paralelo, cada uma em uma subamostra aleatória do dataset (bootstrap), e combina suas predições por **votação majoritária**.

```
Dataset de treino
      │
      ├── Bootstrap 1 → Árvore 1 → predição 1
      ├── Bootstrap 2 → Árvore 2 → predição 2  →  Votação → classe final
      ├── Bootstrap 3 → Árvore 3 → predição 3
      └── ...         → Árvore N → predição N
```

**Dois tipos de aleatoriedade** tornam o RF robusto ao overfitting:
1. **Bagging**: cada árvore treina em uma amostra com reposição (~63% dos dados originais)
2. **Feature Sampling**: em cada divisão de nó, apenas `max_features` features são consideradas — as árvores ficam *decorrelacionadas* entre si

```python
PARAMS_RANDOM_FOREST = {
    'n_estimators'     : [100, 200, 300],   # Quantidade de árvores no ensemble
    'max_depth'        : [None, 10, 20],    # Profundidade máxima de cada árvore
    'min_samples_split': [2, 5],            # Mínimo de amostras para dividir um nó
    'max_features'     : ['sqrt', 'log2'],  # Features por split: sqrt(n) ou log2(n)
}
# Total: 3 × 3 × 2 × 2 = 36 combinações × 5 folds = 180 treinos
```

### 2.1 Random Forest × BoW

In [3]:
print('=' * 55)
print('  Random Forest × BoW')
print('=' * 55)

# n_jobs=-1 dentro do RF usa todos os núcleos para treinar as árvores em paralelo.
# Em BoW com 5.000 features, 'sqrt' seleciona ~70 features por split.
rf_bow = criar_random_forest()
grid_rf_bow = otimizar_modelo(rf_bow, PARAMS_RANDOM_FOREST, X_train_bow, y_train)
salvar_modelo(grid_rf_bow.best_estimator_, '../results/metrics/rf_bow_best.joblib')

  Random Forest × BoW
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Melhores parâmetros : {'max_depth': None, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 100}
Melhor score (f1_weighted): 0.8705
Modelo salvo em: ../results/metrics/rf_bow_best.joblib


### 2.2 Random Forest × TF-IDF

In [4]:
print('=' * 55)
print('  Random Forest × TF-IDF')
print('=' * 55)

# TF-IDF com pesos contínuos — os pontos de corte nos nós exploram melhor
# os valores ponderados do que as contagens inteiras do BoW.
rf_tfidf = criar_random_forest()
grid_rf_tfidf = otimizar_modelo(rf_tfidf, PARAMS_RANDOM_FOREST, X_train_tfidf, y_train)
salvar_modelo(grid_rf_tfidf.best_estimator_, '../results/metrics/rf_tfidf_best.joblib')

  Random Forest × TF-IDF
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Melhores parâmetros : {'max_depth': None, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}
Melhor score (f1_weighted): 0.8784
Modelo salvo em: ../results/metrics/rf_tfidf_best.joblib


### 2.3 Random Forest × Word2Vec

In [5]:
print('=' * 55)
print('  Random Forest × Word2Vec')
print('=' * 55)

# 100 features densas → sqrt(100)=10 features por split.
# O bagging reduz a variância das predições individuais das árvores.
rf_w2v = criar_random_forest()
grid_rf_w2v = otimizar_modelo(rf_w2v, PARAMS_RANDOM_FOREST, para_denso(X_train_w2v), y_train)
salvar_modelo(grid_rf_w2v.best_estimator_, '../results/metrics/rf_w2v_best.joblib')

  Random Forest × Word2Vec
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Melhores parâmetros : {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 300}
Melhor score (f1_weighted): 0.8532
Modelo salvo em: ../results/metrics/rf_w2v_best.joblib


### 2.4 Random Forest × GloVe

In [6]:
print('=' * 55)
print('  Random Forest × GloVe')
print('=' * 55)

# Vetores GloVe pré-treinados NILC-USP: codificam co-ocorrências globais
# do português, produzindo dimensões mais estáveis para os splits das árvores.
rf_glove = criar_random_forest()
grid_rf_glove = otimizar_modelo(rf_glove, PARAMS_RANDOM_FOREST, para_denso(X_train_glove), y_train)
salvar_modelo(grid_rf_glove.best_estimator_, '../results/metrics/rf_glove_best.joblib')

  Random Forest × GloVe
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Melhores parâmetros : {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 300}
Melhor score (f1_weighted): 0.8477
Modelo salvo em: ../results/metrics/rf_glove_best.joblib


## 3. Resumo dos Resultados — Random Forest

In [7]:
resultados_rf = pd.DataFrame([
    {'Modelo': 'Random Forest', 'Representação': 'BoW',      'Melhores Params': grid_rf_bow.best_params_,   'F1-Score (CV)': round(grid_rf_bow.best_score_, 4)},
    {'Modelo': 'Random Forest', 'Representação': 'TF-IDF',   'Melhores Params': grid_rf_tfidf.best_params_, 'F1-Score (CV)': round(grid_rf_tfidf.best_score_, 4)},
    {'Modelo': 'Random Forest', 'Representação': 'Word2Vec', 'Melhores Params': grid_rf_w2v.best_params_,   'F1-Score (CV)': round(grid_rf_w2v.best_score_, 4)},
    {'Modelo': 'Random Forest', 'Representação': 'GloVe',    'Melhores Params': grid_rf_glove.best_params_,  'F1-Score (CV)': round(grid_rf_glove.best_score_, 4)},
])
display(resultados_rf)
resultados_rf.to_csv('../results/metrics/resultados_rf.csv', index=False)
print('✅ resultados_rf.csv salvo em results/metrics/')

,Modelo,Representação,Melhores Params,F1-Score (CV)
0,Random Forest,BoW,"{'max_depth': None, 'max_features': 'log2', 'm...",0.8705
1,Random Forest,TF-IDF,"{'max_depth': None, 'max_features': 'log2', 'm...",0.8784
2,Random Forest,Word2Vec,"{'max_depth': None, 'max_features': 'sqrt', 'm...",0.8532
3,Random Forest,GloVe,"{'max_depth': None, 'max_features': 'sqrt', 'm...",0.8477


✅ resultados_rf.csv salvo em results/metrics/


## 4. AdaBoost

O **AdaBoost** (Adaptive Boosting) é o primeiro algoritmo de Boosting clássico. Ele treina classificadores fracos (**weak learners**, geralmente Decision Stumps — árvores com `max_depth=1`) em **sequência**, aumentando progressivamente o peso das amostras que foram classificadas errado nas iterações anteriores.

```
Iteração 1: Treina Stump 1 → identifica erros → aumenta peso dos erros
Iteração 2: Treina Stump 2 (foca nos erros do 1) → identifica erros → aumenta peso
Iteração 3: Treina Stump 3 (foca nos erros do 2) → ...
        ...
Predição final: soma ponderada de todos os stumps (mais acurados têm mais peso)
```

**Hiperparâmetros otimizados:**
```python
PARAMS_ADABOOST = {
    'n_estimators' : [50, 100, 200],      # Número de stumps (iterações)
    'learning_rate': [0.5, 1.0, 1.5],     # Peso de cada stump na predição final
}
# Total: 3 × 3 = 9 combinações × 5 folds = 45 treinos
```

> **Atenção:** O AdaBoost com representações esparsas (BoW/TF-IDF) pode ser mais lento que o RF porque os stumps são treinados **sequencialmente**, sem paralelização entre iterações.

### 4.1 AdaBoost × BoW

In [8]:
print('=' * 55)
print('  AdaBoost × BoW')
print('=' * 55)

# criar_adaboost() usa algoritmo='SAMME' (discreto), compatível com sklearn >= 1.4.
# O weak learner padrão é um DecisionTreeClassifier(max_depth=1) — um stump.
# Em BoW de alta dimensão, stumps avaliam apenas 1 feature (1 palavra) por split.
ada_bow = criar_adaboost()
grid_ada_bow = otimizar_modelo(ada_bow, PARAMS_ADABOOST, X_train_bow, y_train)
salvar_modelo(grid_ada_bow.best_estimator_, '../results/metrics/ada_bow_best.joblib')

  AdaBoost × BoW
Fitting 5 folds for each of 9 candidates, totalling 45 fits



Melhores parâmetros : {'learning_rate': 1.5, 'n_estimators': 200}
Melhor score (f1_weighted): 0.8048
Modelo salvo em: ../results/metrics/ada_bow_best.joblib


### 4.2 AdaBoost × TF-IDF

In [9]:
print('=' * 55)
print('  AdaBoost × TF-IDF')
print('=' * 55)

# Com TF-IDF, os pontos de corte dos stumps usam valores contínuos de peso IDF,
# o que pode tornar cada stump mais informativo que com BoW (contagens inteiras).
ada_tfidf = criar_adaboost()
grid_ada_tfidf = otimizar_modelo(ada_tfidf, PARAMS_ADABOOST, X_train_tfidf, y_train)
salvar_modelo(grid_ada_tfidf.best_estimator_, '../results/metrics/ada_tfidf_best.joblib')

  AdaBoost × TF-IDF
Fitting 5 folds for each of 9 candidates, totalling 45 fits

Melhores parâmetros : {'learning_rate': 1.5, 'n_estimators': 200}
Melhor score (f1_weighted): 0.8340
Modelo salvo em: ../results/metrics/ada_tfidf_best.joblib


### 4.3 AdaBoost × Word2Vec

In [10]:
print('=' * 55)
print('  AdaBoost × Word2Vec')
print('=' * 55)

# Com embeddings densos de 100 dimensões, cada stump avalia uma dimensão semântica.
# O AdaBoost tende a convergir mais rápido em espaços de baixa dimensão.
ada_w2v = criar_adaboost()
grid_ada_w2v = otimizar_modelo(ada_w2v, PARAMS_ADABOOST, para_denso(X_train_w2v), y_train)
salvar_modelo(grid_ada_w2v.best_estimator_, '../results/metrics/ada_w2v_best.joblib')

  AdaBoost × Word2Vec
Fitting 5 folds for each of 9 candidates, totalling 45 fits

Melhores parâmetros : {'learning_rate': 1.5, 'n_estimators': 200}
Melhor score (f1_weighted): 0.8265
Modelo salvo em: ../results/metrics/ada_w2v_best.joblib


### 4.4 AdaBoost × GloVe

In [11]:
print('=' * 55)
print('  AdaBoost × GloVe')
print('=' * 55)

# Vetores GloVe pré-treinados NILC-USP: as dimensões capturam relações
# semânticas mais ricas do português. Cada stump do AdaBoost irá
# encontrar a dimensão semântica mais discriminativa em cada iteração.
ada_glove = criar_adaboost()
grid_ada_glove = otimizar_modelo(ada_glove, PARAMS_ADABOOST, para_denso(X_train_glove), y_train)
salvar_modelo(grid_ada_glove.best_estimator_, '../results/metrics/ada_glove_best.joblib')

  AdaBoost × GloVe
Fitting 5 folds for each of 9 candidates, totalling 45 fits

Melhores parâmetros : {'learning_rate': 1.0, 'n_estimators': 200}
Melhor score (f1_weighted): 0.8021
Modelo salvo em: ../results/metrics/ada_glove_best.joblib


## 5. Resumo dos Resultados — AdaBoost

In [12]:
resultados_ada = pd.DataFrame([
    {'Modelo': 'AdaBoost', 'Representação': 'BoW',      'Melhores Params': grid_ada_bow.best_params_,   'F1-Score (CV)': round(grid_ada_bow.best_score_, 4)},
    {'Modelo': 'AdaBoost', 'Representação': 'TF-IDF',   'Melhores Params': grid_ada_tfidf.best_params_, 'F1-Score (CV)': round(grid_ada_tfidf.best_score_, 4)},
    {'Modelo': 'AdaBoost', 'Representação': 'Word2Vec', 'Melhores Params': grid_ada_w2v.best_params_,   'F1-Score (CV)': round(grid_ada_w2v.best_score_, 4)},
    {'Modelo': 'AdaBoost', 'Representação': 'GloVe',    'Melhores Params': grid_ada_glove.best_params_,  'F1-Score (CV)': round(grid_ada_glove.best_score_, 4)},
])
display(resultados_ada)
resultados_ada.to_csv('../results/metrics/resultados_ada.csv', index=False)
print('✅ resultados_ada.csv salvo em results/metrics/')

,Modelo,Representação,Melhores Params,F1-Score (CV)
0,AdaBoost,BoW,"{'learning_rate': 1.5, 'n_estimators': 200}",0.8048
1,AdaBoost,TF-IDF,"{'learning_rate': 1.5, 'n_estimators': 200}",0.8340
2,AdaBoost,Word2Vec,"{'learning_rate': 1.5, 'n_estimators': 200}",0.8265
3,AdaBoost,GloVe,"{'learning_rate': 1.0, 'n_estimators': 200}",0.8021


✅ resultados_ada.csv salvo em results/metrics/


## 6. Gradient Boosting

*A implementar*

### 6.1 Gradient Boosting × BoW
### 6.2 Gradient Boosting × TF-IDF
### 6.3 Gradient Boosting × Word2Vec
### 6.4 Gradient Boosting × GloVe

## 7. XGBoost

*A implementar*

### 7.1 XGBoost × BoW
### 7.2 XGBoost × TF-IDF
### 7.3 XGBoost × Word2Vec
### 7.4 XGBoost × GloVe

## 8. Resumo Geral — Todos os Ensembles Base

*A implementar — tabela consolidada (RF + AdaBoost + Gradient Boosting + XGBoost) após treinamento de todos.*

➡️ **Próximo passo:** `06_ensemble_combinados.ipynb` — Voting e Stacking